## MNIST Neural Network from Scratch

### Creating a Dense (Fully connected) layer

In [12]:
import tensorflow as tf

class NaiveDense:
    def __init__(self, input_size, output_size, activation):
        """
        input_size  -> Number of input features
        output_size -> Number of neurons in this layer
        activation  -> Activation function (ReLU, Softmax, etc.)
        """

        # Save the activation function
        self.activation = activation

        # Define the shape of the weight matrix
        w_shape = (input_size, output_size)

        # Initialize weights with small random values
        # Small random values help the neural network start learning.
        w_initial_value = tf.random.uniform(
            shape=w_shape,
            minval=0,
            maxval=1e-1
        )

        # Convert weights into trainable TensorFlow variables
        self.W = tf.Variable(w_initial_value)

        # Create bias vector
        # One bias value for each neuron
        b_shape = (output_size,)

        # Initialize all bias values with zeros
        b_initial_value = tf.zeros(b_shape)

        # Convert biases into trainable TensorFlow variables
        self.b = tf.Variable(b_initial_value)

    def __call__(self, inputs):
        """
        Forward pass of the layer

        Formula:
        output = activation(inputs × weights + bias)
        """

        # Matrix multiplication between inputs and weights
        # Then add bias
        linear_output = tf.matmul(inputs, self.W) + self.b

        # Apply activation function
        return self.activation(linear_output)

    @property
    def weights(self):
        """
        Return trainable parameters of the layer
        """
        return [self.W, self.b]

## A simple sequencial class

In [13]:
class NaiveSequential:
    def __init__(self, layers):
        """
        layers -> List of layers
        """
        self.layers = layers

    def __call__(self, inputs):
        """
        Pass input through all layers sequentially
        """

        # Start with original input
        x = inputs

        # Send output of one layer into next layer
        for layer in self.layers:
            x = layer(x)

        # Final prediction
        return x

    @property
    def weights(self):
        """
        Collect all trainable weights from all layers
        """

        weights = []

        for layer in self.layers:
            weights += layer.weights

        return weights


### Build the Neural Network

In [14]:
model = NaiveSequential([

    # First dense layer
    NaiveDense(
        input_size=28 * 28,   # MNIST image size = 28x28 = 784 pixels
        output_size=512,      # Hidden layer contains 512 neurons
        activation=tf.nn.relu # ReLU activation introduces non-linearity
    ),

    # Output layer
    NaiveDense(
        input_size=512,
        output_size=10,           # 10 output neurons for digits 0-9
        activation=tf.nn.softmax  # Convert outputs into probabilities
    )
])

# Verify total trainable variables
# Layer 1 -> weights + bias
# Layer 2 -> weights + bias
# Total = 4
assert len(model.weights) == 4

### Batch Generator 

In [15]:
import math

class BatchGenerator:
    def __init__(self, images, labels, batch_size=128):
        """
        images     -> Training images
        labels     -> Training labels
        batch_size -> Number of samples per batch
        """

        # Ensure image count and label count are same
        assert len(images) == len(labels)

        # Current batch starting index
        self.index = 0

        # Store dataset
        self.images = images
        self.labels = labels

        # Store batch size
        self.batch_size = batch_size

        # Total number of batches
        self.num_batches = math.ceil(len(images) / batch_size)

    def next(self):
        """
        Return next batch of images and labels
        """

        # Slice images for current batch
        images = self.images[
            self.index : self.index + self.batch_size
        ]

        # Slice labels for current batch
        labels = self.labels[
            self.index : self.index + self.batch_size
        ]

        # Move index forward
        self.index += self.batch_size

        # Return current batch
        return images, labels

### Optimizer

In [16]:
from tensorflow.keras import optimizers

# SGD = Stochastic Gradient Descent
optimizer = optimizers.SGD(learning_rate=1e-3)


def update_weights(gradients, weights):
    """
    Update model weights using computed gradients
    """

    # Apply gradients to corresponding weights
    optimizer.apply_gradients(zip(gradients, weights))

### Per batch Training loop

In [17]:
def one_training_step(model, images_batch, labels_batch):
    """
    Perform one forward pass + backward pass
    """

    # GradientTape records operations for automatic differentiation
    with tf.GradientTape() as tape:

        # Forward pass
        predictions = model(images_batch)

        # Compute loss for each sample
        per_sample_losses = tf.keras.losses.sparse_categorical_crossentropy(
            labels_batch,
            predictions
        )

        # Compute average batch loss
        average_loss = tf.reduce_mean(per_sample_losses)

    # Compute gradients of loss with respect to model weights
    gradients = tape.gradient(average_loss, model.weights)

    # Update weights using gradients
    update_weights(gradients, model.weights)

    # Return batch loss
    return average_loss

### Full training loop of an epoch

In [18]:
def fit(model, images, labels, epochs, batch_size=128):
    """
    Train the model for multiple epochs
    """

    # Loop through epochs
    for epoch_counter in range(epochs):

        print(f"Epoch {epoch_counter + 1}")

        # Create batch generator
        batch_generator = BatchGenerator(
            images,
            labels,
            batch_size=batch_size
        )

        # Iterate through all batches
        for batch_counter in range(batch_generator.num_batches):

            # Get next batch
            images_batch, labels_batch = batch_generator.next()

            # Perform one training step
            loss = one_training_step(
                model,
                images_batch,
                labels_batch
            )

            # Print loss every 100 batches
            if batch_counter % 100 == 0:
                print(f"Loss at batch {batch_counter}: {loss:.4f}")


### Test it

In [19]:
# Load MNIST Dataset
from tensorflow.keras.datasets import mnist

# Load training and testing data
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()


# Preprocess Training Data

# Flatten 28x28 images into 784-dimensional vectors
train_images = train_images.reshape((60000, 28 * 28))

# Convert integer pixels into float32
train_images = train_images.astype("float32")

# Normalize pixel values from [0,255] → [0,1]
train_images = train_images / 255


# Preprocess Test Data

# Flatten test images
test_images = test_images.reshape((10000, 28 * 28))

# Convert to float32
test_images = test_images.astype("float32")

# Normalize
test_images = test_images / 255


# Train the Model
fit(
    model,
    train_images,
    train_labels,
    epochs=10,
    batch_size=128
)

Epoch 1
Loss at batch 0: 6.9340
Loss at batch 100: 2.2404
Loss at batch 200: 2.1981
Loss at batch 300: 2.0741
Loss at batch 400: 2.2356
Epoch 2
Loss at batch 0: 1.8937
Loss at batch 100: 1.8854
Loss at batch 200: 1.8187
Loss at batch 300: 1.6976
Loss at batch 400: 1.8464
Epoch 3
Loss at batch 0: 1.5691
Loss at batch 100: 1.5853
Loss at batch 200: 1.4912
Loss at batch 300: 1.4139
Loss at batch 400: 1.5276
Epoch 4
Loss at batch 0: 1.3129
Loss at batch 100: 1.3416
Loss at batch 200: 1.2276
Loss at batch 300: 1.1977
Loss at batch 400: 1.2919
Epoch 5
Loss at batch 0: 1.1161
Loss at batch 100: 1.1582
Loss at batch 200: 1.0365
Loss at batch 300: 1.0363
Loss at batch 400: 1.1236
Epoch 6
Loss at batch 0: 0.9708
Loss at batch 100: 1.0196
Loss at batch 200: 0.8999
Loss at batch 300: 0.9170
Loss at batch 400: 1.0011
Epoch 7
Loss at batch 0: 0.8629
Loss at batch 100: 0.9131
Loss at batch 200: 0.7987
Loss at batch 300: 0.8275
Loss at batch 400: 0.9100
Epoch 8
Loss at batch 0: 0.7813
Loss at batch 10

### Evaluating the model

In [20]:
# Generate predictions on test data
predictions = model(test_images)

# Convert TensorFlow tensor into NumPy array
predictions = predictions.numpy()

# Import NumPy
import numpy as np

# Select class with highest probability
predicted_labels = np.argmax(predictions, axis=1)

# Compare predicted labels with true labels
matches = predicted_labels == test_labels

# Compute average accuracy
accuracy = matches.mean()

# Print final accuracy
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8171
